# Example 11.3 / 2: Trace What the Solver Receives

Companion notebook for **11.3 Nonlinear Expressions and Functions** in the AMPL lab Documentation, section "Trace What the Solver Receives". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Trace What the Solver Receives

AMPL stores each expression as a tree of operators and operands (an expression tree), and an
MP-based driver, the interface between AMPL and a solver, processes those trees according to the
solver's capabilities and settings. The same source expression can therefore reach different
solvers by different routes:

| Route | What it means | What to verify | Example |
|---|---|---|---|
| native expression | the driver passes a supported nonlinear or quadratic form | solver capability and local or global guarantee | `x^2 + y^2 <= 1` with Ipopt reaches the exact optimum $-1.41421$ (Quadratic Models (AMPL lab documentation, section 11.1)) |
| exact reformulation | auxiliary variables and constraints preserve the modeled relation | required domains and the generated formulation | `minimize abs(x - 3) + 0.1*x` with HiGHS, run below: `x = 3`, objective `0.3`, no approximation warning |
| conic recognition | an algebraic pattern is transmitted as a supported cone | sign restrictions, cone direction, and solver-model export | `sqrt(sum {i in ITEMS} Mix[i]^2) <= Radius` reaches Gurobi as a second-order cone (Second-Order Cones (AMPL lab documentation, section 11.5)) |
| piecewise-linear approximation | a bounded-domain approximation replaces the original function | argument bounds, error policy, and residuals in the original function | `minimize exp(x) - 2*x` with HiGHS, run below: warning `PLApprox_Exp`; `x = 0.6087` instead of $\log 2=0.693147$ |
| rejection | no supported native or converted representation is available | choose another solver or an equivalent justified formulation | an imported GSL function with HiGHS fails with `Not handling constraint type 'Call'` (Imported Functions and GSL (AMPL lab documentation, section 11.3.1)) |

HiGHS has no `abs`, yet the MP driver rewrites it exactly with linear constraints, so the
returned point is the true minimizer:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -10, <= 10;  # finite bounds help the reformulation

minimize Obj: abs(x - 3) + 0.1 * x;  # nonsmooth but convex; exact optimum at x = 3

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["x"].value(), round(ampl.obj["Obj"].value(), 6))

solved 3.0 0.3
